Hour of day - Time in the raw dataset is seconds elapsed since the first transaction. Fraud patterns differ at 3am vs 3pm. Extracting the hour gives the model a cyclical signal it couldn't see in raw seconds.

Log amount - Amount is heavily right-skewed (most transactions are small, a few are huge). Log-transforming it compresses the scale and makes the distribution more normal, which helps both logistic regression and tree models.

Amount z-score - how many standard deviations above or below the mean is this transaction? A z-score of +4 flags an unusually large amount, which is a classic fraud signal.

Amount/mean ratio - similar idea, expressed as a multiple of the average transaction. Slightly more interpretable than z-score in a business context.

# Before SMOTE

In [4]:
import numpy as np
import joblib
import os

os.makedirs('data/features', exist_ok=True)

X_train = joblib.load('data/smote/X_train_resampled.pkl')
X_test  = joblib.load('data/preprocessed/X_test.pkl')

def add_features(df, amount_mean, amount_std):
    df = df.copy()
    df['Hour']         = (df['Time'] % 86400) / 3600
    df['Log_Amount']   = np.log1p(df['Amount'])
    df['Amount_Zscore']= (df['Amount'] - amount_mean) / amount_std
    df['Amount_Ratio'] = df['Amount'] / amount_mean
    return df

amount_mean = X_train['Amount'].mean()
amount_std  = X_train['Amount'].std()

X_train = add_features(X_train, amount_mean, amount_std)
X_test  = add_features(X_test,  amount_mean, amount_std)

joblib.dump(X_train, 'data/features/X_train_resampled.pkl')
joblib.dump(X_test,  'data/features/X_test.pkl')
joblib.dump({'amount_mean': amount_mean, 'amount_std': amount_std},
            'data/feature_stats.pkl')

print(f"Feature count: {X_train.shape[1]}")  # 34
print(X_train[['Hour','Log_Amount','Amount_Zscore','Amount_Ratio']].describe())

Feature count: 34
               Hour     Log_Amount  Amount_Zscore   Amount_Ratio
count  4.549020e+05  454902.000000   4.549020e+05  454902.000000
mean   1.407237e+01      -0.104978  -2.834653e-15       1.000000
std    1.181946e+01       0.451607   1.000000e+00      22.647028
min    5.301281e-09      -0.433385  -4.246381e-01      -8.616791
25%    2.979021e-04      -0.417093  -4.131170e-01      -8.355872
50%    2.399961e+01      -0.306448  -3.297077e-01      -6.466900
75%    2.399979e+01       0.046042   6.820416e-03       1.154462
max    2.400000e+01       4.635864   1.104331e+02    2501.982449


Instead of 227k synthetic samples per class, we have 394 vs 394 a tiny but perfectly balanced dataset. The feature engineering code stays identical, we just swap the file paths:

# After SMOTE


In [5]:
# Load undersampled train set (not the SMOTE one)
X_train = joblib.load('data/smote/X_train_under.pkl')
y_train = joblib.load('data/smote/y_train_under.pkl')
X_test  = joblib.load('data/preprocessed/X_test.pkl')

def add_features(df, amount_mean, amount_std):
    df = df.copy()
    df['Hour']          = (df['Time'] % 86400) / 3600
    df['Log_Amount']    = np.log1p(df['Amount'])
    df['Amount_Zscore'] = (df['Amount'] - amount_mean) / amount_std
    df['Amount_Ratio']  = df['Amount'] / amount_mean
    return df

# Compute stats from training set only
amount_mean = X_train['Amount'].mean()
amount_std  = X_train['Amount'].std()

X_train = add_features(X_train, amount_mean, amount_std)
X_test  = add_features(X_test,  amount_mean, amount_std)

# Save with clear undersampling names so nothing gets mixed up
joblib.dump(X_train, 'data/features/X_train_under_eng.pkl')
joblib.dump(X_test,  'data/features/X_test_eng.pkl')
joblib.dump({'amount_mean': amount_mean, 'amount_std': amount_std},
            'data/features/feature_stats.pkl')

print(f"Train shape: {X_train.shape}")   # (788, 34)
print(f"Test shape:  {X_test.shape}")    # should be ~56962 rows
print(f"Class balance:\n{y_train.value_counts()}")

Train shape: (788, 34)
Test shape:  (56962, 34)
Class balance:
0    394
1    394
Name: Class, dtype: int64
